<a href="https://colab.research.google.com/github/wjohn564/AlcoGuardian/blob/main/20245424_EA_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div>
<img src="https://www.ul.ie/themes/custom/ul/logo.jpg" />
</div>

#**MSc in Artificial Intelligence and Machine Learning**
##CS6271 - Evolutionary Algorithms and Humanoid Robotics 2025
### Kaggle Competition


Module Leader: Conor Ryan

Developer: John Walsh

Predict whether a person will earn more than 50k or less. This is a modified version of the adult dataset

In [ ]:
# Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Dataset

**Class:**

output: 1, 0.


**Listing of features:**

- 'age'
- 'workclass'
- 'fnlwgt'
- 'education'
- 'education-num'
- 'marital-status'
- 'occupation'
- 'relationship'
- 'race'
- 'sex'
- 'capital-gain'
- 'capital-loss'
- 'hours-per-week'
- 'native-country'

In [ ]:
# Suppressing Warnings:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
## mount your Google drive
# 1) run this cell
# 2) sign in
# 3) verify your drive is mounted

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## Grape Setup for Grammatical Evolution

In [ ]:
import os
# Get the library from our BDS research Group
# copy the path from your drive
PATH = '/content/drive/MyDrive/grape/'

# check if 'grape' already exists
if os.path.exists(PATH):
    print('grape directory already exists')
else:
    %cd /content/drive/MyDrive/
    !git clone https://github.com/bdsul/grape.git
    print('Cloning grape in your Drive')

# change directory to 'grape'
%cd /content/drive/MyDrive/grape/

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



grape directory already exists
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipython-input-1217348620.py", line 15, in <cell line: 0>
    get_ipython().run_line_magic('cd', '/content/drive/MyDrive/grape/')
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2418, in run_line_magic
    result = fn(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^
  File "<decorator-gen-85>", line 2, in cd
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magic.py", line 187, in <lambda>
    call = lambda f, *a, **k: f(*a, **k)
                              ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magics/osm.py", line 342, in cd
    oldcwd = os.getcwd()
             ^^^^^^^^^^^
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above excep

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipython-input-1217348620.py", line 15, in <cell line: 0>
    get_ipython().run_line_magic('cd', '/content/drive/MyDrive/grape/')
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2418, in run_line_magic
    result = fn(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^
  File "<decorator-gen-85>", line 2, in cd
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magic.py", line 187, in <lambda>
    call = lambda f, *a, **k: f(*a, **k)
                              ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magics/osm.py", line 342, in cd
    oldcwd = os.getcwd()
             ^^^^^^^^^^^
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occurre

## Load the training dataset, visualise some stats

In [ ]:
# Load the Training Set
train_file = '/content/train.csv'
df_train = pd.read_csv(train_file)

# Show the first 5 rows
df_train.head()

In [ ]:
# Visualise some helpful details
df_train.describe()

In [ ]:
# More stats to view
df_train.describe(include='object')

# Load the Test dataset (**NO LABELS**), visualise some stats and create the target variable

In [ ]:
# Load the Test Set

train_file = '/content/test.csv'
df_test = pd.read_csv(train_file)

# Show the first 5 rows
df_test.head()

In [ ]:
# Visualise some helpful details
df_test.describe()

In [ ]:
# More stats to view
df_test.describe(include='object')

# Data Preprocessing


Some Helper Functions

In [ ]:
from sklearn.preprocessing import LabelEncoder

def add_label_encoder_to_columns(df, categorical_cols):
  """
  This function will be used on the training data to map integars to the
  Categorical columns. It does this by fitting a label encoder onto each categorical
  column.

  The function returns a dictionary: column_name: encoder
  """
  encoders = {}

  for col in categorical_cols:
    # instantiate the LabelEncoder class
    le = LabelEncoder()
    # give NaN type values 'Unknown'
    types = df[col].fillna('Unknown').astype(str)
    # create mappings for categories to numeric values
    le.fit(types)
    # add the encoder to the dictionary
    encoders[col] = le
  return encoders


def apply_label_encoder(df, encoders, categorical_cols, numerical_cols):
  """
  This function will apply any previously defined encoder to the
  supplied dataframe.

  returns a fully numerical dataframe
  """
  # Create a copy to work with
  df = df.copy()

  # Encode the categorical columns
  for col in categorical_cols:
    le = encoders[col]
    types = df[col].fillna('Unknown').astype(str)

    # quick check to make sure 'Unknown' is a new encoder class
    if 'Unknown' not in le.classes_:
      le.classes_ = np.append(le.classes_, 'Unknown')

    # check for categories not picked up from initial mapping
    # assign 'unknown' to those types
    mask_unknown = ~np.isin(types, le.classes_)
    if mask_unknown.any():
      types[mask_unknown] = 'Unknown'

    # finally apply the encoding
    df[col] = le.transform(types)


  # convert numeric cols to floats
  for col in numerical_cols:
    df[col] = df[col].astype(float)

  # concatinate them together
  feature_cols = numerical_cols + categorical_cols
  return df[feature_cols]



Applying Encoders

In [ ]:
# List the categorical cols
categorical_cols = ['workclass', 'education', 'marital-status', 'occupation',
                    'relationship', 'race', 'sex', 'native-country'
                    ]

# List the numerical cols
numerical_cols = ['age', 'fnlwgt', 'education-num', 'capital-gain',
                  'capital-loss', 'hours-per-week'
                  ]

# remove target variable from training data and create copies of datasets to
# work with
y = df_train['income']
X_train = df_train.drop(columns=['income'])
X_test = df_test.copy()

# Use the function above to fit an encoder on the training set
encoders = add_label_encoder_to_columns(X_train, categorical_cols)

# Apply the encoder to the training and test sets
X_train = apply_label_encoder(X_train, encoders, categorical_cols, numerical_cols)
X_test = apply_label_encoder(X_test, encoders, categorical_cols, numerical_cols)

print(f"Encoded training set shape: {X_train.shape}")
print(f"Encoded test set shape: {X_test.shape}")

Final touch up for grape

In [ ]:
# put in numpy array format
Y_train = y.to_numpy()

# Applied the transpose as per the comment given
X_train = np.transpose(X_train)
X_test = np.transpose(X_test)

### NB***** make sure this cell doesnt run again or it will apply another transpose
# and then it will not be correct
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape} ")
print(f"Y_train: {Y_train.shape}")

## GRAPE

<div>
<img src="https://drive.google.com/uc?export=view&id=1hw43Oi3lGTCkspQ0ged2bZB8q2EpcPhz" width="150"/>
</div>

GRammatical Algorithms in Python for Evolution (GRAPE)

In [ ]:
!pip install deap

import grape
import algorithms

from os import path
from deap import creator, base, tools
import random
import csv

Import the functions to be used with the grammar from [functions.py](https://github.com/UL-BDS/grape/blob/main/functions.py) int the GRAPE repository

In [ ]:
from functions import and_, or_, not_, less_than_or_equal, greater_than_or_equal

Create a bnf file with the Grammar

Note: The grammar style I am going with is inspired by Conor Ryans github repo: [HERE](https://github.com/conorlime/CS6271)


In [ ]:
# This cell is to create Grammar and save it as a bnf file

# The Feature indices:
# 0: age
# 1: fnlwgt
# 2: education-num
# 3: capital-gain
# 4: capital-loss
# 5: hours-per-week
# 6: workclass
# 7: education (this one is the categorical feature)
# 8: marital-status
# 9: occupation
# 10: relationship
# 11: race
# 12: sex
# 13: native-country

adult_grammar = r"""
<log_op> ::= <conditional>
           | and_(<log_op>, <log_op>)
           | or_(<log_op>, <log_op>)
           | not_(<log_op>)

<conditional> ::= greater_than_or_equal(<feature>, <constant>)
                | less_than_or_equal(<feature>, <constant>)

<feature> ::= x[0]  | x[1]  | x[2]  | x[3]  | x[4]  | x[5]
            | x[6]  | x[7]  | x[8]  | x[9]  | x[10] | x[11]
            | x[12] | x[13]

<constant> ::= 0 | 5 | 10 | 15 | 20 | 25 | 30 | 35 | 40 | 45
             | 50 | 55 | 60 | 65 | 70 | 75 | 80
"""

with open("/content/adult_income.bnf", "w") as f:
    f.write(adult_grammar)

In [ ]:
GRAMMAR_FILE = "/content/adult_income.bnf"
# Open and display Grammar
f = open( GRAMMAR_FILE, "r")
print(f.read())
f.close()

The following cell puts the grammar on the class Grammar.

In [ ]:
BNF_GRAMMAR = grape.Grammar(GRAMMAR_FILE)

I am using the supplied fitness function which tracks the percentage of outputs wrongly predicted.

In [ ]:
def fitness_eval(individual, points):
    """
    Fitness Function
    """

    x = points[0]
    Y = points[1]

    if individual.invalid == True:
        return np.nan,

    # Evaluate the expression
    try:
        pred = eval(individual.phenotype)
    except (FloatingPointError, ZeroDivisionError, OverflowError,
            MemoryError):
        return np.nan,
    assert np.isrealobj(pred)

    compare = np.equal(Y,pred)
    fitness = 1 - np.mean(compare)

    return fitness,

Grammatical Evolution parameters.

In [ ]:
POPULATION_SIZE = 400
MAX_GENERATIONS = 60
P_CROSSOVER = 0.9
P_MUTATION = 0.1
ELITE_SIZE = 1
HALL_OF_FAME_SIZE = 1

TOURNAMENT_SIZE = 4
RANDOM_SEED = 43
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# How codons are consumed and stored
CODON_CONSUMPTION = 'lazy'
GENOME_REPRESENTATION = 'list'
CODON_SIZE = 255
MAX_GENOME_LENGTH = 100

# Random initialisation
MIN_INIT_GENOME_LENGTH = 30
MAX_INIT_GENOME_LENGTH = 80

# Controls for tree depth
MAX_INIT_TREE_DEPTH = 8
MAX_TREE_DEPTH      = 12
MAX_WRAPS           = 0

REPORT_ITEMS = ['gen', 'invalid', 'avg', 'std', 'min', 'max',
                'best_ind_length', 'avg_length',
                'best_ind_nodes', 'avg_nodes',
                'best_ind_depth', 'avg_depth',
                'avg_used_codons', 'best_ind_used_codons',
                'structural_diversity', 'fitness_diversity',
                'selection_time', 'generation_time']

In [ ]:
toolbox = base.Toolbox()

# define a single objective, minimising fitness strategy:
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))

creator.create('Individual', grape.Individual, fitness=creator.FitnessMin)

toolbox.register("populationCreator", grape.random_initialisation, creator.Individual)

toolbox.register("evaluate", fitness_eval)

# Tournament selection:
toolbox.register("select", tools.selTournament, tournsize=TOURNAMENT_SIZE)

# Single-point crossover:
toolbox.register("mate", grape.crossover_onepoint)

# Flip-int mutation:
toolbox.register("mutate", grape.mutation_int_flip_per_codon)

In [ ]:
# create initial population (generation 0):
population = toolbox.populationCreator(
    pop_size=POPULATION_SIZE,
    bnf_grammar=BNF_GRAMMAR,
    min_init_genome_length=MIN_INIT_GENOME_LENGTH,
    max_init_genome_length=MAX_INIT_GENOME_LENGTH,
    max_init_depth=MAX_INIT_TREE_DEPTH,
    codon_size=CODON_SIZE,
    codon_consumption=CODON_CONSUMPTION,
    genome_representation=GENOME_REPRESENTATION,
)

# define the hall-of-fame object:
hof = tools.HallOfFame(HALL_OF_FAME_SIZE)

# prepare the statistics object:
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("avg", np.nanmean)
stats.register("std", np.nanstd)
stats.register("min", np.nanmin)
stats.register("max", np.nanmax)

Run GE

In [ ]:
population, logbook = algorithms.ge_eaSimpleWithElitism(population, toolbox, cxpb=P_CROSSOVER, mutpb=P_MUTATION,
                                              ngen=MAX_GENERATIONS, elite_size=ELITE_SIZE,
                                              bnf_grammar=BNF_GRAMMAR,
                                              codon_size=CODON_SIZE,
                                              max_tree_depth=MAX_TREE_DEPTH,
                                              max_genome_length=MAX_GENOME_LENGTH,
                                              points_train=[X_train, Y_train],
                                              codon_consumption=CODON_CONSUMPTION,
                                              report_items=REPORT_ITEMS,
                                              genome_representation=GENOME_REPRESENTATION,
                                              stats=stats, halloffame=hof, verbose=False)

show the best individual as an expression

In [ ]:
# Best individual
import textwrap
best = hof.items[0].phenotype
print("Best individual: \n","\n".join(textwrap.wrap(best,80)))
print("\nTraining Fitness: ", hof.items[0].fitness.values[0])

Define a function to predict values, without comparing to expected outputs.

In [ ]:
def predict(individual, X):
    x = X

    if individual.invalid == True:
        return np.NaN,

    # Evaluate the expression
    try:
        pred = eval(individual.phenotype)
    except (FloatingPointError, ZeroDivisionError, OverflowError,
            MemoryError):
        return np.NaN,
    assert np.isrealobj(pred)

    return pred

Predict the classes of the test set.

Make sure you print here in the notebook you will submit to Brightspace the same predictions you used in your best submission to the Kaggle competition.

In [ ]:
y_pred = predict(hof.items[0], X_test)
print("Predicted classes of the test set: ", y_pred)

Write a code to create a .csv with the following format:
1. First column is the index (from 0 to 95803);
2. Second column is named `output` and contains the predictions (only 0's or 1's) you  got in the previous cell with y_pred.

Example:

    index,output

    0,0

    1,0

    2,1

    ...

    95803,0


Submit it to the competition and check your score there.